In [2]:
import pandas as pd
from dae_finder import PolyFeatureMatrix
import Comparison

Model Beer  
Original model  

$\frac{dx_1}{dt} = -\frac{\beta*x_1*H(t-tau)*(x_1-Bacmax)}{Bacmax}$  
$\frac{dx_2}{dt} = -k_{syn}*x_1x_2$  

$\frac{dx_3}{dt} = k_{syn}*x_1x_2 - k_{dim}*x_3^2$  
$\frac{dx_4}{dt} = k_{dim}*x_3^2 - k_{degi}*x_4$

Parameter (log 10) 

$Bacmax$: -0.2034  
$\beta$: -2.15448  
$k_{degi}$: -1.98745  
$k_{dim}$: -4.16399  
$k_{syn}$: -2.25703

In [6]:
# Load model Beer
Beer_model = pd.read_excel('Model_beer.xlsx')
# Load time series data
data_beer = pd.read_excel('Beer_exp.xlsx')
data_beer.rename(columns={
    'x_Exp_1': 'x1',
    'x_Exp_2': 'x2',
    'x_Exp_3': 'x3',
    'x_Exp_4': 'x4'
},inplace=True)
time_beer = data_beer['t_Exp'].to_numpy()
cols = [col for col in data_beer.columns if col not in ['t_Exp']]
#data_states = data[cols].to_numpy()
data_states_beer = data_beer.copy().drop(columns={'t_Exp'})
data_derivatives_beer = Comparison.compute_time_derivatives(data_beer,'t_Exp',method='spline')

In [10]:
# Create function libarary and generate corresponding values for each degree
degree_range = [2,3,4,5]
candidate_libs_beer = {}
recover_beer = {}
condition_number_beer,Missing_beer, Wrong_beer = {},{},{}
# original model
Beer = Comparison.Terms_Identification(Beer_model)
variable_mapping,all_xi_terms = Beer._parse_model()
for poly_degree in degree_range:
    poly_feature_ob = PolyFeatureMatrix(poly_degree)

    candidate_lib_full = poly_feature_ob.fit_transform(data_states_beer)


    # Dropping 1 from the candidate library since we use fit_with_intercept flag instead of using constants directly
    # in the candidate library
    candidate_lib_full = candidate_lib_full.drop(["1"], axis=1)
    candidate_libs_beer[poly_degree] = candidate_lib_full
    # DAE_finder
    recover = Comparison.Recover_Model(candidate_lib_full,data_derivatives_beer,threshold=1e-5)
    recovered_model,recovered_mapping = recover.model,recover.mapping
    recover_beer[poly_degree] = recover

    Analyzer_Beer = Comparison.Terms_Analysis(Beer_model,recovered_model,variable_mapping,recovered_mapping,candidate_lib_full)
    condition_number, wrong, missing = Analyzer_Beer.con, Analyzer_Beer.wrong_terms, Analyzer_Beer.missing_terms
    condition_number_beer[poly_degree], Wrong_beer[poly_degree], Missing_beer[poly_degree] = condition_number, wrong, missing

Sequential threshold converged in 0 iterations
Sequential threshold converged in 0 iterations
Sequential threshold converged in 0 iterations
Sequential threshold converged in 1 iterations
Sequential threshold converged in 1 iterations
Sequential threshold converged in 1 iterations
Sequential threshold converged in 2 iterations
Sequential threshold converged in 1 iterations
Sequential threshold converged in 4 iterations
Sequential threshold converged in 1 iterations
Sequential threshold converged in 1 iterations
Sequential threshold converged in 4 iterations
Sequential threshold converged in 3 iterations
Sequential threshold converged in 4 iterations
Sequential threshold converged in 4 iterations
Sequential threshold converged in 1 iterations


In [11]:
# Noise Free Results
degree_list = [1,2,3,4,5]
comb_list = [2,3]
summary_beer = Comparison.run_noise_free_analysis(data_beer, degree_list, comb_list)
summary_beer

>>> Running noise-free analysis...
>>> Generating candidate libraries...
    Degrees: [1, 2, 3, 4, 5]
    Combinations: [2, 3]


,degree,# lib terms,# ill-posed 2comb,# ill-posed 3comb
0,1,4,0,1
1,2,14,2,104
2,3,34,14,1397
3,4,69,60,10164
4,5,125,205,56342
